In [ ]:
#ファイルフォルダー_マウント
from google.colab import drive
import os

#個人ドライブの場合のパス
#PATH_RAW = "/content/drive/MyDrive/〇〇"
#共有ドライブの場合のパス
#PATH_RAW = "/content/drive/Shareddrives/〇〇"

#個人ドライブ直下「data_raw」フォルダー配下をファイル走査先に設定
base_path = "/content/drive/MyDrive/data-raw"

In [ ]:
#ファイル_走査

import chardet
import unicodedata
import pandas as pd

raw_path = []

#ファイル_パスジョイン
for root, dirs, files in os.walk(base_path):

  for file in files:
    file_normalized = unicodedata.normalize("NFC", file)
    file_lower = file.lower()

    #エクセルの場合
    if file_lower.endswith(".xlsx"):
      excel_path = os.path.join(root, file)
      raw_path.append(excel_path)

    #CSVの場合
    elif file_lower.endswith(".csv"):
      csv_path = os.path.join(root, file)
      raw_path.append(csv_path)

    #CSVの場合_文字エンコード変換
    try:
      with open(csv_path, 'rb') as c:
        csv_file = chardet.detect(c.read())
        encode = csv_file.get("encoding")

        #UTF-8の場合
        if encode and encode.lower() == 'utf-8':
          csv_df = pd.read_csv(csv_path, encoding='utf-8')
          print(f"{file}を読み込みました")
        #UTF-16の場合
        elif encode and encode.lower() == 'utf-16':
          csv_df = pd.read_csv(csv_path, encoding='utf-16')
          csv_df.to_csv(csv_path, index=False, encoding='utf-8')
          print(f"{file}（{encode}）のUTF-16をUTF-8に変換・上書き保存・読み込みました")
        #Shift-JIS・cp932の場合
        else:
          csv_df = pd.read_csv(csv_path, encoding='cp932')
          csv_df.to_csv(csv_path, index=False, encoding='utf-8')
          print(f"{file}（{encode}）のShift-JIS・cp932をUTF-8に変換・上書き保存・読み込みました")

    except Exception as e:
      print(f"エラー：{file}の処理に問題が発生しました - {e}")

In [ ]:
#sampleファイル_読み込み

from pathlib import Path

#データ内容=[0], テーブル名=[1], ファイル名=[2]にタプル
sample_file = []

for file in raw_path:
  if "sample" in file.lower():

    #エクセルの場合
    if file.endswith(".xlsx"):
      excel_file = pd.ExcelFile(file)

      for sheet in excel_file.sheet_names:
        excel_df = excel_file.parse(sheet)
        table = f"{Path(file).stem}_{sheet}"
        sample_file.append((excel_df, table, file))
        print(f"ファイル名：{file}｜シート名：{sheet} \n{excel_df.head(2)}\n")

    #CSVの場合
    elif file.endswith(".csv"):
      csv_df = pd.read_csv(file)
      table = Path(file).stem
      sample_file.append((csv_df, table, file))
      print(f"ファイル名：{file}｜シート名：（なし） \n{csv_df.head(2)}\n")

    else:
      #条件に合致しなかった場合のデバッグ用
      print(f"エラー：{file}の処理をスキップしました")

In [ ]:
#sampleデータ_読み込み

import re

sample_df = []

#sampleデータ_クレンジング_①ファイル名から属性列を付与・補完
for df, table, file in sample_file:
  df = df.assign(属性1="sample")

  #ファイル名から日付列を付与・補完の場合
  #（ファイル名例：sample_YYYYMMDD.xlsx など）
  #sampleデータには、日付列が存在するため割愛
  #↓
  #month_search = re.search(r"_\d{8}", file)
  #month_extract = month_search.group() if month_search else None
  #df = df.assign(属性1="sample", 日付=month_extract)

  sample_df.append((df, table, file))

import sqlite3
memory_conn = sqlite3.connect(":memory:")

for df, table, file in sample_df:
  df.to_sql(table, memory_conn, index=False)
  print(f"ファイル名：{file}｜シート名：{table} \n{df.head(2)}\n")

In [ ]:
#sampleデータ_クレンジング_②見出し列の整理
queries = '''
CREATE TABLE テーブル1 AS

SELECT
日付 AS 中間列_日付,
属性1,
属性2,
属性3,
"1_v" AS 指標1,
"2_v" AS 指標2,
"3_v" AS 指標3,
"4_v" AS 指標4
FROM
"sample_202601"

UNION
SELECT
日付 AS 中間列_日付,
属性1,
属性2,
属性3,
"1_v" AS 指標1,
"2_v" AS 指標2,
"3_v" AS 指標3,
"4_v" AS 指標4
FROM
"sample_202602"

UNION
SELECT
日付 AS 中間列_日付,
属性1,
属性2,
属性3,
"1_v" AS 指標1,
"2_v" AS 指標2,
"3_v" AS 指標3,
"4_v" AS 指標4
FROM
"sample_202603_シート1"
'''

#sampleデータ_統合版としてシート1へ一時保存
memory_conn.execute("DROP TABLE IF EXISTS テーブル1;")
memory_conn.execute(queries)
clean_df = pd.read_sql("SELECT * FROM テーブル1", memory_conn)

print(f"ファイル名：テーブル1 \n{clean_df.head(2)}\n")

In [ ]:
#sampleデータ_クレンジング_③フィールドの整理
queries = [
    "ALTER TABLE テーブル1 ADD COLUMN 日付 text;",
    "UPDATE テーブル1 SET 日付 = strftime('%Y/%m/%d', 中間列_日付);",

    "ALTER TABLE テーブル1 ADD COLUMN 属性4_性別 text;",
    """
    UPDATE テーブル1
    SET 属性4_性別 = CASE
        WHEN 属性2 LIKE 'AA%' THEN '男性'
        WHEN 属性2 LIKE 'BK%' THEN '女性'
        ELSE 'その他'
    END;
    """,

    "ALTER TABLE テーブル1 ADD COLUMN 属性5_地域 text;",
    """
    UPDATE テーブル1
    SET 属性5_地域 = CASE
        WHEN 属性3 LIKE '%_5_%' THEN '東京'
        WHEN 属性3 LIKE '%_6_%' THEN '神奈川'
        WHEN 属性3 LIKE '%_7_%' THEN '埼玉'
        WHEN 属性3 LIKE '%_8_%' THEN '千葉'
        WHEN 属性3 LIKE '%_9_%' THEN '大阪'
        WHEN 属性3 LIKE '%_10_%' THEN '京都'
        WHEN 属性3 LIKE '%_11_%' THEN '兵庫'
        ELSE 'その他'
    END;
    """,
]

for query in queries:
    try:
        memory_conn.execute(query)
    except sqlite3.OperationalError as e:
        #列がすでに存在する場合はスキップ
        if "duplicate column name" in str(e):
            pass
        else:
            #正規化出来なかった場合のデバッグ用
            print(f"エラー：{query}の処理をスキップしました: {e}")

#sampleデータ_統合版としてシート1へ一時保存（追記）
clean_df = pd.read_sql("SELECT * FROM テーブル1", memory_conn)

print(f"テーブル名：テーブル1 \n{clean_df.head(2)}\n")

In [ ]:
#sampleデータ_統合版の保存_①Googleスプレッドシートの場合_指定

import gspread
from google.colab import auth
from google.auth import default
from gspread_dataframe import set_with_dataframe

#Googleアカウント_認証
#↓
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

#Googleスプレッドシート_指定（ファイル名の場合）
#↓
sh = gc.open("スプレッドシートのファイル名")

#Googleスプレッドシート_指定（URLの場合）
#↓
spreadsheet_url = "[スプレッドシートのURL]"
sh = gc.open_by_url(spreadsheet_url)

#書き込み
try:
    worksheet = sh.worksheet("テーブル1")
    既存データのクリア
    worksheet.clear()
except gspread.exceptions.WorksheetNotFound:
    worksheet = sh.add_worksheet(title="テーブル1", rows="100", cols="20")

#分割して書き込み
chunk_size = 1000 #分割する行数
total_rows = len(clean_df)

print(f"総行数 {total_rows} 行のデータをスプレッドシートへ分割書き込み中")

for i in range(0, total_rows, chunk_size):
    chunk_df = clean_df.iloc[i : i + chunk_size]

    #書き込み1回目（i=0）のブロックだけヘッダーを含める
    include_header = True if i == 0 else False

    #書き込み開始行（1回目の処理は1行目から、2回目以降はデータ行の続きから）
    start_row = 1 if i == 0 else (i + 1)

    set_with_dataframe(
        worksheet,
        chunk_df,
        row=start_row,
        include_index=False,
        include_column_header=include_header,
   )

    print(
        f"-> {i} 行目から {min(i + chunk_size, total_rows)} 行目までの書き込みが完了"
   )

    #次のブロックがある場合のみ、書き込み制限回避のために休憩を入れる
    if i + chunk_size < total_rows:
        time.sleep(4)

print("\nGoogleスプレッドシートへの書き込みが正常に完了")

In [ ]:
#sampleデータ_統合版の保存_②.dbファイルの場合_作成

#個人ドライブ直下「data_processed」フォルダーを書き出し先に設定
clean_db = '/content/drive/MyDrive/data_processed'

if os.path.exists(clean_db):
    print("ファイル名.dbの書き込みが正常に完了")
else:
    print("ファイル名.dbの新規作成・書き込みが正常に完了")

clean_df = pd.read_sql("SELECT * FROM テーブル1", memory_conn)

#書き込み
clean_conn = sqlite3.connect(clean_db)
clean_df.to_sql("テーブル1", clean_conn, if_exists="replace", index=False)
clean_conn.close()

print(f"テーブル名：テーブル1 \n{clean_df.head(2)}\n")

In [ ]:
#sampleデータ_統合版の保存_③BigQueryの場合
#要4点事前準備（プロジェクト、データセット、テーブル、サービスアカウントJSONファイル）

import os
#自身でエクスポートしたサービスアカウントJSONファイルを指定
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/drive/MyDrive/サービスアカウントJSONファイル.json"

from google.cloud import bigquery

client = bigquery.Client(project="プロジェクト名")
dataset_table = "プロジェクト名.データセット名.テーブル名"

job_config = bigquery.LoadJobConfig(write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE)
job = client.load_table_from_dataframe(clean_df, dataset_table, job_config=job_config)
job.result()

print(len(clean_df), "行の書き込みが正常に完了")